# Technical Walkthrough — Predictive Log Anomaly Engine V3

This notebook explains the internal architecture and execution flow of the repository in engineering terms. It is not a product demo notebook.

## Context
- The system ingests log events and produces anomaly/alert decisions.
- V1/V2/V3 are layered evolution stages in this repository:
  - V1: baseline token/window anomaly pipeline in the core runtime path.
  - V2: dedicated multi-model ML pipeline (`src/runtime/pipeline_v2.py`) exposed under `/v2/*`.
  - V3: optional semantic explanation/enrichment layer (`src/semantic/*`) gated by environment flags.

Covered in this notebook: internal data flow, key modules, runtime behavior, and operational constraints.

## 2. High-Level Architecture

Core runtime modules:
- API layer: `src/api`
- Runtime orchestration: `src/runtime`
- Modeling code: `src/modeling`
- Alert lifecycle: `src/alerts`
- Observability: `src/observability`
- Semantic layer (V3): `src/semantic`

Text architecture sketch:

Client -> FastAPI (`src/api/app.py`) -> Pipeline (`src/api/pipeline.py`)
-> Inference engine (`src/runtime/inference_engine.py`)
-> Alert policy/manager (`src/alerts/manager.py`)
-> Outbox/Webhook (`src/alerts/n8n_client.py`)
-> Metrics (`/metrics`, `src/observability/metrics.py`)

## 3. End-to-End Flow (Critical Section)

Pipeline shape:

Log -> Parsing/Normalization -> Template/Token context -> Sequence window -> Model score -> Threshold decision -> Alert policy

In the active API path, events are processed via `Pipeline.process_event()` in `src/api/pipeline.py`, which delegates to `InferenceEngine.ingest()` in `src/runtime/inference_engine.py`.

In [1]:
%pip install -e ..

import sys
from pathlib import Path

# Ensure repo root is importable in this kernel session
repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Example usage pattern (requires configured environment and artifacts)
from src.api.pipeline import Pipeline

pipeline = Pipeline()
event = {
    "message": "error connecting to DB",
    "level": "ERROR",
    "service": "api",
    "session_id": "demo-session",
    "timestamp": 0.0,
    "token_id": 42,
    "label": 0,
}

result = pipeline.process_event(event)
print(result)

Obtaining file:///C:/Users/ORENS/predictive-log-anomaly-engine-v3
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for UNKNOWN (pyproject.toml): started
  Building editable for UNKNOWN (pyproject.toml): finished with status 'done'
  Created wheel for UNKNOWN: filename=unknown-0.0.0-0.editable-py3-none-any.whl size=1259 sha256=30d56b5a35763c7e038c4caf969e7fed376e6f6d7c1c3f2cd1bb8b4739b528bc
  Stored in directory: C:\Users\ORENS\AppData\Local\Temp\pip-ephem-wheel-cache-89ak5kq2\wheels\ba\6d\0a\0e74e4229845975812456cd95220b70011f3

Stage-by-stage interpretation of `result`:
- `window_emitted=False`: rolling buffer has not reached an emission boundary yet.
- `risk_result`: present once a full window is emitted and scored.
- `alert`: present only if score + policy + cooldown produce a fired alert.

Relevant implementation points:
- Buffer and scoring: `src/runtime/sequence_buffer.py`, `src/runtime/inference_engine.py`
- Risk result typing: `src/runtime/types.py`
- Alert emission: `src/alerts/manager.py`

## 4. V1 vs V2 vs V3 Comparison

### V1 (baseline/token-window path)
- Main runtime path used by `/ingest` in `src/api/routes.py`.
- Implemented in `src/runtime/inference_engine.py`.
- Uses baseline and/or transformer scoring modes: `baseline`, `transformer`, `ensemble`.
- Controlled by `MODEL_MODE` in `src/api/settings.py`.

### V2 (multi-model ML pipeline path)
- Exposed by `/v2/ingest` and `/v2/alerts` in `src/api/routes_v2.py`.
- Core orchestration in `src/runtime/pipeline_v2.py`.
- API wrapper with cooldown/ring buffer in `src/runtime/inference_engine_v2.py`.
- Startup wiring in `src/api/app.py` when `MODEL_MODE` contains `v2`.

### V3 (semantic enrichment layer)
- Additive layer attached in `src/api/pipeline.py` after anomaly confirmation.
- Config from `src/semantic/config.py` and env flags (`SEMANTIC_ENABLED`, `EXPLANATION_ENABLED`).
- Versioned endpoints in `src/api/routes_v3.py` for model info and explanations.

## 5. Runtime Behavior Deep Dive

### `src/runtime/inference_engine.py`
Key behaviors:
- Maintains per-stream rolling windows via `SequenceBuffer`.
- Loads runtime artifacts (`artifacts/vocab.json`, `templates.json`, threshold files).
- Loads baseline and/or transformer models based on configured mode.
- Produces a structured `RiskResult` with score, threshold, evidence window, and metadata.

Threshold behavior:
- Baseline and transformer thresholds are loaded from artifact JSON files.
- Optional runtime override file: `artifacts/threshold_runtime.json` (when enabled).
- Ensemble decision normalizes score space against original threshold denominators.

### `src/runtime/pipeline_v2.py`
V2 processing chain:
1. Raw log normalization and token mapping via `_V2LogTokenizer`.
2. Word2Vec lookup for token IDs.
3. Rolling embedding window construction.
4. `SystemBehaviorModel` context modeling.
5. `AnomalyDetector` reconstruction-error score.
6. `SeverityClassifier` output (`info`/`warning`/`critical`).

Important compatibility note from config/docs in code:
- V2 expects a window size consistent with training assumptions (commonly `WINDOW_SIZE=10`).

## 6. Alerting Logic

Alert generation flow:
- `Pipeline.process_event()` receives a `RiskResult` from runtime.
- `AlertManager.emit()` evaluates anomaly policy + threshold + cooldown.
- On fire, alert is buffered in memory and sent to `N8nWebhookClient`.

Dedup/cooldown:
- Managed per `stream_key` in `src/alerts/manager.py`.
- `ALERT_COOLDOWN_SECONDS` controls suppression interval.

n8n integration mode:
- Safe default is dry-run (writes JSON to `artifacts/n8n_outbox/`).
- Live webhook mode requires `N8N_DRY_RUN=false` and a configured `N8N_WEBHOOK_URL`.

In [2]:
# Conceptual alert extraction from a processed event
from src.api.pipeline import Pipeline

pipeline = Pipeline()
sample = {
    "service": "hdfs",
    "session_id": "blk_42",
    "timestamp": 0.0,
    "token_id": 123,
    "label": 0,
}
out = pipeline.process_event(sample)

if out.get("alert"):
    print("Alert fired:", out["alert"]["alert_id"], out["alert"]["severity"])
else:
    print("No alert fired (window/policy/cooldown conditions not met yet).")

No alert fired (window/policy/cooldown conditions not met yet).


## 7. Observability

Observability wiring:
- Metrics primitives are defined in `src/observability/metrics.py` (`Counter`, `Histogram`, `Gauge`).
- API exposes `/metrics` in `src/api/routes.py`.
- `MetricsMiddleware` captures ingest latency.
- Health status is reflected in a gauge (`service_health`).

Infra linkage:
- Prometheus scrape config: `prometheus/prometheus.yml`
- Grafana provisioning/dashboards: `grafana/provisioning/`, `grafana/dashboards/`
- Compose wiring: `docker/docker-compose.yml`

## 8. Configuration & Environment

Configuration sources:
- Runtime defaults + env parsing: `src/api/settings.py`
- Example variables: `.env.example`

Important flags and effects:
- `MODEL_MODE`: selects scoring mode and optional v2 startup path.
- `WINDOW_SIZE`, `STRIDE`: window emission behavior.
- `METRICS_ENABLED`: enables metrics registry + endpoint behavior.
- `DISABLE_AUTH`, `API_KEY`, `PUBLIC_ENDPOINTS`: request auth behavior.
- `ALERT_COOLDOWN_SECONDS`: per-stream suppression control.
- `SEMANTIC_ENABLED`, `EXPLANATION_ENABLED`: V3 semantic activation.

Operational recommendation:
- Treat `.env.example` as contract documentation for deployment/runtime setup.

## 9. Known Limitations

Current technical limitations visible from repository behavior:
- Native-library dependencies (`pandas`, `torch`, etc.) are environment-sensitive and may be blocked by host policies.
- Full test collection can fail in constrained local environments even when containerized runtime paths are healthy.
- V2 path depends on trained artifacts and expected dimensional assumptions.
- Semantic layer is optional and disabled by default to keep base runtime lightweight.
- Alert buffer is in-memory; persistence beyond process lifetime requires external storage integration.

## 10. Summary

The codebase is modular and layered:
- API transport and lifecycle are isolated in `src/api`.
- Runtime inference logic is isolated in `src/runtime`.
- Alert policy/integration is isolated in `src/alerts`.
- Observability is first-class via Prometheus instrumentation.
- V3 semantic enrichment is additive and feature-gated.

This architecture supports conservative evolution: new behavior can be added through versioned routes and optional layers without rewriting the main runtime path.